In [ ]:
!pip install -q "numpy==1.26.4"
!pip install -q "lightning-utilities"
!pip install -q "pytorch-lightning==1.9.5" "torchmetrics==0.11.4" --no-deps
!pip install -q "scikit-learn" "timm" "python-box" "grad-cam" "ttach" "setuptools==69.5.1"

In [ ]:
import os
print(os.listdir())

import os

print(os.listdir("/kaggle/input"))

print(os.listdir("/content"))

import numpy as np
np.Inf = np.inf

In [ ]:
import numpy as np
print("numpy version:", np.__version__)
# Must print 1.26.4 before continuing
assert np.__version__ == "1.26.4", "Wrong numpy! Restart and re-run install cell"

In [ ]:
import os
import warnings
from pprint import pprint
from glob import glob
from tqdm import tqdm

import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchvision.transforms as T
from box import Box
from timm import create_model
from sklearn.model_selection import StratifiedKFold
from torchvision.io import read_image
from torch.utils.data import DataLoader, Dataset
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

import pytorch_lightning as pl
from pytorch_lightning.utilities.seed import seed_everything
from pytorch_lightning.callbacks import ProgressBarBase, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning import LightningDataModule, LightningModule

# Patch for older PyTorch Lightning versions that expect np.Inf
if not hasattr(np, 'Inf'):
    np.Inf = np.inf

warnings.filterwarnings("ignore")

In [ ]:
config = {'seed': 2021,
          'root': '/content/drive/MyDrive',
          'n_splits': 3,
          'epoch': 10,
          'trainer': {
            'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu',
            'devices': 1,
            'accumulate_grad_batches': 1,
            'fast_dev_run': False,
            'num_sanity_val_steps': 0,
            'max_epochs': 20,
          },
          'transform':{
              'name': 'get_default_transforms',
              'image_size': 224
          },
          'train_loader':{
              'batch_size': 64,
              'shuffle': True,
              'num_workers': 4,
              'pin_memory': False,
              'drop_last': True,
          },
          'val_loader': {
              'batch_size': 64,
              'shuffle': False,
              'num_workers': 4,
              'pin_memory': False,
              'drop_last': False
         },
          'model':{
              'name': 'swin_tiny_patch4_window7_224',
              'output_dim': 1
          },
          'optimizer':{
              'name': 'optim.AdamW',
              'params':{
                  'lr': 1e-5
              },
          },
          'scheduler':{
              'name': 'optim.lr_scheduler.CosineAnnealingWarmRestarts',
              'params':{
                  'T_0': 20,
                  'eta_min': 1e-4,
              }
          },
          'loss': 'nn.BCEWithLogitsLoss',
}

config = Box(config)

In [ ]:
pprint(config)

In [ ]:
class PetfinderDataset(Dataset):
    def __init__(self, df, image_size=224):
        self._X = df["Id"].values
        self._y = None
        if "Pawpularity" in df.keys():
            self._y = df["Pawpularity"].values
        self._transform = T.Resize([image_size, image_size])

    def __len__(self):
        return len(self._X)

    def __getitem__(self, idx):
        image_path = self._X[idx]
        image = read_image(image_path)
        image = self._transform(image)
        if self._y is not None:
            label = self._y[idx]
            return image, label
        return image

class PetfinderDataModule(LightningDataModule):
    def __init__(
        self,
        train_df,
        val_df,
        cfg,
    ):
        super().__init__()
        self._train_df = train_df
        self._val_df = val_df
        self._cfg = cfg

    def __create_dataset(self, train=True):
        return (
            PetfinderDataset(self._train_df, self._cfg.transform.image_size)
            if train
            else PetfinderDataset(self._val_df, self._cfg.transform.image_size)
        )

    def train_dataloader(self):
        dataset = self.__create_dataset(True)
        return DataLoader(dataset, **self._cfg.train_loader)

    def val_dataloader(self):
        dataset = self.__create_dataset(False)
        return DataLoader(dataset, **self._cfg.val_loader)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
torch.autograd.set_detect_anomaly(True)
seed_everything(config.seed)

from google.colab import files
files.upload()
df = pd.read_csv("train.csv")
df["Id"] = df["Id"].apply(lambda x: os.path.join(config.root, "train", x + ".jpg"))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive

In [ ]:
import shutil
import os

src = "/content/drive/MyDrive/train"  # just the train folder
dst = "/content/train"

if not os.path.exists(dst):
    shutil.copytree(src, dst)
    print("Done copying")
else:
    print("Already exists, skipping")

print(os.listdir("/content/train")[:5])

In [ ]:
!ls /content
!ls /content/train | head

In [ ]:
sample_dataloader = PetfinderDataModule(df, df, config).val_dataloader()
images, labels = next(iter(sample_dataloader))

plt.figure(figsize=(12, 12))
for it, (image, label) in enumerate(zip(images[:16], labels[:16])):
    plt.subplot(4, 4, it + 1)
    plt.imshow(image.permute(1, 2, 0))
    plt.axis('off')
    plt.title(f'Pawpularity: {int(label)}')
plt.suptitle('Sample images from dataset', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

feature_cols = ['Subject Focus', 'Eyes', 'Face', 'Near', 'Action',
                'Accessory', 'Group', 'Collage', 'Human', 'Occlusion',
                'Info', 'Blur', 'Pawpularity']

corr = df[feature_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5
)
plt.title('Feature correlation heatmap')
plt.tight_layout()
plt.show()

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]     #RGB
IMAGENET_STD = [0.229, 0.224, 0.225]      #RGB

def get_default_transforms():
    return {
        "train": T.Compose([
            T.Resize((224, 224)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomAffine(15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            T.ColorJitter(0.1, 0.1, 0.1),
            T.ConvertImageDtype(torch.float),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]),
        "val": T.Compose([
            T.Resize((224, 224)),
            T.ConvertImageDtype(torch.float),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]),
    }

In [ ]:
class Model(pl.LightningModule):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.__build_model()
        self._criterion = eval(self.cfg.loss)()
        self.transform = get_default_transforms()
        self.save_hyperparameters(cfg)

    def __build_model(self):
        self.backbone = create_model(
            self.cfg.model.name, pretrained=True, num_classes=0, in_chans=3
        )
        num_features = self.backbone.num_features
        self.fc = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(num_features, self.cfg.model.output_dim)
        )

    def forward(self, x):
        f = self.backbone(x)
        out = self.fc(f)
        return out

    def training_step(self, batch, batch_idx):
        loss, pred, labels = self.__share_step(batch, 'train')
        return {'loss': loss, 'pred': pred, 'labels': labels}

    def validation_step(self, batch, batch_idx):
        loss, pred, labels = self.__share_step(batch, 'val')
        return {'pred': pred, 'labels': labels}

    def __share_step(self, batch, mode):
        images, labels = batch
        labels = labels.float() / 100.0
        images = self.transform[mode](images)

        logits = self.forward(images).squeeze(1)
        loss = self._criterion(logits, labels)

        pred = logits.sigmoid().detach().cpu() * 100.
        labels = labels.detach().cpu() * 100.
        return loss, pred, labels

    def training_epoch_end(self, outputs):
        self.__share_epoch_end(outputs, 'train')

    def validation_epoch_end(self, outputs):
        self.__share_epoch_end(outputs, 'val')

    def __share_epoch_end(self, outputs, mode):
        preds = []
        labels = []
        for out in outputs:
            pred, label = out['pred'], out['labels']
            preds.append(pred)
            labels.append(label)
        preds = torch.cat(preds)
        labels = torch.cat(labels)
        metrics = torch.sqrt(((labels - preds) ** 2).mean())
        self.log(f'{mode}_loss', metrics)

    def check_gradcam(self, dataloader, target_layer, target_category, reshape_transform=None):
        cam = GradCAMPlusPlus(
            model=self,
            target_layers=[target_layer],  # now a list
            reshape_transform=reshape_transform
        )

        org_images, labels = next(iter(dataloader))
        cam.batch_size = len(org_images)
        images = self.transform['val'](org_images)
        images = images.to(self.device)
        logits = self.forward(images).squeeze(1)
        pred = logits.sigmoid().detach().cpu().numpy() * 100
        labels = labels.cpu().numpy()

        grayscale_cam = cam(input_tensor=images, targets=None, eigen_smooth=True)
        org_images = org_images.detach().cpu().numpy().transpose(0, 2, 3, 1) / 255.
        return org_images, grayscale_cam, pred, labels

    def configure_optimizers(self):
        optimizer = eval(self.cfg.optimizer.name)(
            self.parameters(), **self.cfg.optimizer.params
        )
        scheduler = eval(self.cfg.scheduler.name)(
            optimizer, **self.cfg.scheduler.params
        )
        return [optimizer], [scheduler]

In [ ]:
# Ensure consistency in config
config.epoch = 10  # or whatever you want

config.trainer = dict(
    accelerator="auto",
    devices=1,
    precision=16,
    max_epochs=config.epoch
)

In [ ]:
# 1. Define the Logger
logger = TensorBoardLogger("logs", name="pawpularity_model")

# 2. Define the Callbacks
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min"
)

lr_monitor = pl.callbacks.LearningRateMonitor(logging_interval='step')

checkpoint = pl.callbacks.ModelCheckpoint(
    monitor='val_loss',
    save_top_k=1,
    save_last=True,
    mode='min'
)

# 3. Now run the Trainer
trainer = pl.Trainer(
    logger=logger,
    callbacks=[early_stopping, lr_monitor, checkpoint],
    enable_progress_bar=True,
    **config.trainer  # max_epochs is already in here
)

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from sklearn.model_selection import StratifiedKFold

# ----------------------------
# K-Fold Setup
# ----------------------------
skf = StratifiedKFold(
    n_splits=config.n_splits,
    shuffle=True,
    random_state=config.seed
)

# ----------------------------
# Training Loop
# ----------------------------
for fold, (train_idx, val_idx) in enumerate(skf.split(df["Id"], df["Pawpularity"])):

    print(f"\n🚀 Starting Fold {fold}")

    # Split data
    train_df = df.loc[train_idx].reset_index(drop=True)
    val_df = df.loc[val_idx].reset_index(drop=True)

    # DataModule + Model
    datamodule = PetfinderDataModule(train_df, val_df, config)
    model = Model(config)

    # ----------------------------
    # Callbacks
    # ----------------------------
    early_stopping = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3
    )

    lr_monitor = LearningRateMonitor(logging_interval='epoch')

    checkpoint = ModelCheckpoint(
        filename=f"best_loss_fold_{fold}",
        monitor="val_loss",
        save_top_k=1,
        mode="min"
    )

    # ----------------------------
    # Logger
    # ----------------------------
    logger = TensorBoardLogger(
        save_dir="tb_logs",
        name=config.model.name,
        version=f"fold_{fold}"
    )

    # ----------------------------
    # Trainer (✅ FIXED + INDENTED)
    # ----------------------------
    trainer_config = config.trainer.copy()

    trainer = pl.Trainer(
        logger=logger,
        callbacks=[early_stopping, lr_monitor, checkpoint],
        enable_progress_bar=True,
        **trainer_config
    )

    # ----------------------------
    # Train
    # ----------------------------
    trainer.fit(model, datamodule=datamodule)

In [ ]:
import glob
ckpts = glob.glob("**/*.ckpt", recursive=True)
print(ckpts)

In [ ]:
def reshape_transform(tensor):
    print("tensor shape:", tensor.shape)
    if tensor.dim() == 4:
        return tensor
    b, seq, c = tensor.shape
    h = w = int(seq ** 0.5)
    result = tensor.reshape(b, h, w, c)
    result = result.permute(0, 3, 1, 2)
    return result

model = Model(config)
model.load_state_dict(torch.load(
    'tb_logs/swin_tiny_patch4_window7_224/fold_0/checkpoints/best_loss_fold_0.ckpt',
    weights_only=False
)['state_dict'])
model = model.cuda().eval()

config.val_loader.batch_size = 16
datamodule = PetfinderDataModule(train_df, val_df, config)
images, grayscale_cams, preds, labels = model.check_gradcam(
    datamodule.val_dataloader(),
    target_layer=model.backbone.layers[-1].blocks[-1].norm1,
    target_category=None,
    reshape_transform=reshape_transform
)

plt.figure(figsize=(12, 12))
for it, (image, grayscale_cam, pred, label) in enumerate(zip(images[:16], grayscale_cams[:16], preds[:16], labels[:16])):
    plt.subplot(4, 4, it + 1)
    visualization = show_cam_on_image(image, grayscale_cam)
    plt.imshow(visualization)
    plt.title(f'pred: {pred:.1f} label: {label}')
    plt.axis('off')
plt.show()

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from glob import glob

path = glob('tb_logs/swin_tiny_patch4_window7_224/fold_0/events*')[0]
event_acc = EventAccumulator(path, size_guidance={'scalars': 0})
event_acc.Reload()

scalars = {}
for tag in event_acc.Tags()['scalars']:
    events = event_acc.Scalars(tag)
    scalars[tag] = [event.value for event in events]

print("Available tags:", list(scalars.keys()))

plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plt.plot(scalars['train_loss'], label='train_loss')
plt.plot(scalars['val_loss'], label='val_loss')
plt.legend()
plt.ylabel('rmse')
plt.xlabel('epoch')
plt.title('train/val rmse')
plt.show()

print('best_val_loss:', min(scalars['val_loss']))